# Drug Discovery GRPO — Kaggle Runbook

End-to-end on a Kaggle GPU notebook (T4 x1 / x2 or P100):

1. Clone the repo from GitHub
2. Install all dependencies (TRL, Unsloth 4-bit, RDKit, FastAPI, etc.)
3. Build the disease/target dataset (`prepare_dataset.py`)
4. Boot the FastAPI env server inside the notebook
5. Run multi-disease GRPO training (`train.py`)
6. Evaluate on the held-out test split (`evaluate.py`)
7. Run inference on a brand-new (unseen) disease (`infer.py`)

> Before running: in **Kaggle Notebook → Settings**, enable **Internet** and pick an
> **Accelerator** (`GPU T4 x2` recommended). Then run the cells in order.

## 0. Choose run knobs

These are the only values you should typically change. They are exported as env
vars and consumed by the CLI flags below. Defaults are tuned for a Kaggle T4.

In [ ]:
import os

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/VasuBB/drug-discovery-sim-env.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
WORK_DIR = '/kaggle/working/drug-discovery-sim-env' if os.path.isdir('/kaggle') else os.path.abspath('drug-discovery-sim-env')

# Dataset prep
NUM_DISEASES = int(os.environ.get('NUM_DISEASES', '600'))    # bump to 5000+ for the full dataset
TEST_FRACTION = float(os.environ.get('TEST_FRACTION', '0.10'))
MIN_DRUGGABILITY = float(os.environ.get('MIN_DRUGGABILITY', '0.30'))
KNOWN_DRUGS_PER_TARGET = int(os.environ.get('KNOWN_DRUGS_PER_TARGET', '8'))

# Training
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
NUM_TRAIN_STEPS = int(os.environ.get('NUM_TRAIN_STEPS', '50'))
GROUP_SIZE = int(os.environ.get('GROUP_SIZE', '4'))
ENV_PORT = int(os.environ.get('ENV_PORT', '8000'))
BASE_URL = f'http://127.0.0.1:{ENV_PORT}'

# Evaluation / inference
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '20'))          # cap test diseases (None = use all)
INFER_DISEASE = os.environ.get('INFER_DISEASE', 'Idiopathic pulmonary fibrosis')

print('Work dir:', WORK_DIR)
print('Model:', MODEL_NAME, '| GRPO steps:', NUM_TRAIN_STEPS, '| group size:', GROUP_SIZE)

## 1. Clone the repo

In [ ]:
import os, subprocess, sys

if not os.path.isdir(WORK_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, WORK_DIR])
else:
    subprocess.check_call(['git', '-C', WORK_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working in', os.getcwd())

## 2. Install dependencies

Unsloth ships its own torch/triton wheels, so we install it first and let pip
resolve everything else around it. RDKit and TRL come from the `[chem]` and
`[training]` extras.

In [ ]:
%%capture
!pip install --quiet --upgrade pip
!pip install --quiet 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' || true
!pip install --quiet 'trl>=0.11' 'transformers>=4.44' 'accelerate>=0.33' 'datasets>=2.20' 'peft>=0.11' 'bitsandbytes>=0.43'
!pip install --quiet 'rdkit>=2024.3.1' rank-bm25 'sentence-transformers>=2.7'
!pip install --quiet 'fastapi>=0.115' 'uvicorn[standard]>=0.30' 'pydantic>=2.7' 'pydantic-settings>=2.2' 'PyYAML>=6.0' requests numpy networkx
!pip install --quiet -e .[training,test,chem]

In [ ]:
import importlib, torch
for m in ['transformers', 'trl', 'datasets', 'peft', 'accelerate', 'rdkit', 'fastapi', 'uvicorn', 'drug_discovery_env']:
    try:
        mod = importlib.import_module(m)
        print(f'{m}: OK ({getattr(mod, "__version__", "")})')
    except Exception as e:
        print(f'{m}: FAIL -', e)
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Build the disease / target / known-drugs dataset

One-time fetch from Open Targets + ChEMBL. Set `NUM_DISEASES` higher (e.g. 5500)
for a full run. Internet must be enabled.

In [ ]:
!python -m drug_discovery_env.scripts.prepare_dataset \
    --num-diseases {NUM_DISEASES} \
    --test-fraction {TEST_FRACTION} \
    --min-druggability {MIN_DRUGGABILITY} \
    --known-drugs-per-target {KNOWN_DRUGS_PER_TARGET}

import json, pathlib
manifest = json.loads(pathlib.Path('data/diseases.manifest.json').read_text())
manifest

## 4. Boot the FastAPI env server (background)

We launch uvicorn in a subprocess and wait for `/health` to return 200 before
moving on. Logs are streamed to `outputs/server.log`.

In [ ]:
import os, subprocess, time, requests

os.makedirs('outputs', exist_ok=True)
log_handle = open('outputs/server.log', 'w', buffering=1)
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'drug_discovery_env.server.app:app',
     '--host', '127.0.0.1', '--port', str(ENV_PORT), '--log-level', 'warning'],
    stdout=log_handle, stderr=subprocess.STDOUT,
)

for attempt in range(60):
    try:
        r = requests.get(f'{BASE_URL}/health', timeout=2.0)
        if r.status_code == 200:
            print('env server up:', r.json())
            break
    except Exception:
        pass
    time.sleep(1.0)
else:
    raise RuntimeError('env server failed to start; check outputs/server.log')

## 5. GRPO training

Multi-disease live-rollout training. Per-turn JSONL traces (tool calls,
literature queries, reasoning, reward breakdown) are written under
`outputs/grpo/logs/<run-id>/`.

In [ ]:
!python -m drug_discovery_env.scripts.train \
    --base-url {BASE_URL} \
    --model {MODEL_NAME} \
    --num-train-steps {NUM_TRAIN_STEPS} \
    --group-size {GROUP_SIZE} \
    --output-dir outputs/grpo \
    --log-dir outputs/grpo/logs \
    --run-id kaggle-run

In [ ]:
import pathlib
for p in sorted(pathlib.Path('outputs/grpo').glob('*'))[:20]:
    print(p)
print('---')
csv_path = pathlib.Path('outputs/grpo/logs/kaggle-run/runs.csv')
if csv_path.exists():
    print(csv_path.read_text()[:2000])

## 6. Evaluate on the held-out test split

Computes the full panel: env reward, ADMET pass, oversight violations, budget
remaining, mean reasoning depth, and ChEMBL Tanimoto-to-known-drugs (precision@1
vs. cached known compounds for each test disease's target).

In [ ]:
!python -m drug_discovery_env.scripts.evaluate \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --limit {EVAL_LIMIT} \
    --run-id kaggle-eval

In [ ]:
import json, pathlib
report = json.loads(pathlib.Path('outputs/eval/report.json').read_text())
report

## 7. Inference on an unseen disease

Pass any disease string. If it isn't in the cache the env falls back to live
Open Targets for the target lookup, then drives a full campaign with the
trained checkpoint.

In [ ]:
!python -m drug_discovery_env.scripts.infer \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --disease "{INFER_DISEASE}" \
    --out-dir outputs/infer

import json, pathlib, re
slug = re.sub(r'[^a-z0-9]+', '-', INFER_DISEASE.lower()).strip('-')
summary = json.loads(pathlib.Path(f'outputs/infer/{slug}.json').read_text())
summary

## 8. Shut down the env server

In [ ]:
try:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print('env server stopped (exit', server_proc.returncode, ')')
except Exception as exc:
    print('shutdown error:', exc)
    server_proc.kill()

## Outputs

Everything below is plain JSON / JSONL and can be downloaded from the Kaggle
notebook output panel:

- `data/diseases.jsonl` — cached disease/target/known-drug rows + `train`/`test` split
- `data/diseases.manifest.json` — row counts, sha256, fetch timestamp
- `outputs/grpo/` — trained checkpoint (HF format, or PEFT adapter)
- `outputs/grpo/logs/kaggle-run/*.jsonl` — per-episode turn-level traces
- `outputs/grpo/logs/kaggle-run/runs.csv` — one-line summary per training episode
- `outputs/eval/report.json` — aggregate evaluation panel
- `outputs/eval/per_disease.jsonl` — per-test-disease metrics
- `outputs/infer/<slug>.json` — inference summary + reasoning trace